# LAB 4 - MD Analysis
**What did CLN025 do? From trajectories to free energies**


Authors:

- Prof. Marco A. Deriu (marco.deriu@polito.it)
- Eric A. Zizzi (eric.zizzi@polito.it)
- Marcello Miceli (marcello.miceli@polito.it)

# Table of Contents

1. The data
2. RMSD: similar to what?
3. Flexibility and size: RMSF and radius of gyration
4. Backbone angles: the Ramachandran plot
5. Hydrogen bonds
6. Secondary structure
7. Observables of the hairpin: end-to-end distance, aromatic core, native contacts
8. Many runs: the class dataset
9. Principal component analysis
10. Free-energy surfaces
11. The equilibrium free-energy profile along the end-to-end distance
12. Clustering the conformations
13. Your deliverable

**Learning outcomes:**
- turn a trajectory into numbers with GROMACS tools and Python
- choose the reference of an RMSD, and explain why the choice matters
- measure the observables that define the CLN025 hairpin
- pool many runs, and say what they can and cannot tell
- compute a free-energy surface and a one-dimensional free-energy profile, with error bars
- group conformations into clusters and recognise folded and misfolded states

# 1. The data

This lab analyses two sets of runs, all at 300 K:

| Set | What | From |
|---|---|---|
| **Your LAB 3 run** | NMR model 1, 10 ns | Your LAB 3 folder, or the course's copy: `course-fetch --lab 03-ClassicalMD --solutions` |
| **Class dataset** | One 10 ns run for each of the 20 NMR models of 2RVD | Your class's runs of LAB 3 exercise 8.1, or the course's copy: `course-fetch --lab 04-Analysis --solutions` |

Sections 2 to 7 analyse your LAB 3 run; sections 8 to 12 pool the class dataset.

Every analysis needs the same two files per run: the trajectory (`.xtc`) and the run input file (`.tpr`). To keep things fast and small, the first step keeps only the peptide.

Choose where the LAB 3 run is: `../03-ClassicalMD` for your own, `../03-ClassicalMD/solutions` for the course's.

In [ ]:
%env LAB3=../03-ClassicalMD

## 1.1 Keep the peptide, make it whole

`trjconv` writes the peptide alone, whole and centred (as at the end of LAB 3), `convert-tpr` writes a run input file with only the peptide's atoms, and a first frame is saved as `.pdb` for Python. `gmx select` writes an index file with the groups used below, defined on residues 1–10 so that the caps are left out:

In [ ]:
%%bash
mkdir -p runs results
src=$LAB3/cln025_model1/04-md/md_300K
cd runs
printf "Protein\nProtein\n" | gmx trjconv -s ../$src.tpr -f ../$src.xtc -o md_300K.xtc -pbc mol -center > prepare.out 2>&1 || tail -n 20 prepare.out
printf "Protein\nProtein\n" | gmx trjconv -s ../$src.tpr -f ../$src.xtc -o md_300K.pdb -pbc mol -center -dump 0 >> prepare.out 2>&1
echo Protein | gmx convert-tpr -s ../$src.tpr -o md_300K.tpr >> prepare.out 2>&1
gmx select -s md_300K.tpr -on md_300K.ndx \
    -select '"Protein" group "Protein"' '"CA" resid 1 to 10 and name CA' '"Backbone" resid 1 to 10 and name N CA C' >> prepare.out 2>&1
echo "$(grep -c '^ATOM' md_300K.pdb) atoms"
ls

From here on, every command reads `runs/md_300K.xtc` and `runs/md_300K.tpr`.

The course's fixed definitions (RMSD, native contacts, end-to-end distance, what counts as folded) are in `../common/scripts/cln025.py`, so that LABs 4, 5 and 6 use exactly the same ones. Import it, with a reader for `.xvg` files:

In [ ]:
import sys
import inspect
import numpy as np
import matplotlib.pyplot as plt

sys.path.append("../common/scripts")
import cln025
from cln025 import xvg

# 2. RMSD: similar to what?

The root-mean-square deviation measures how far a structure is from a **reference**, after the best superposition of the two:

$$RMSD(t) = \sqrt{\frac{1}{N}\sum_{i=1}^{N}\left|r_i(t) - r_i^{ref}\right|^2}$$

where $N$ is the number of atoms compared: here the backbone atoms (N, Cα, C) of residues 1–10.

An RMSD is meaningless without its reference. A run whose RMSD from the start "stabilises" at 0.3 nm may be visiting many structures, all 0.3 nm from the start but far from each other. Compare the run with three references: its first frame, its last frame, and its **average structure**, which `gmx rmsf -ox` writes:

In [ ]:
%%bash
cd runs
echo Protein | gmx trjconv -s md_300K.tpr -f md_300K.xtc -o last.gro -dump 1000000 > last.out 2>&1 || tail -n 20 last.out
echo Protein | gmx rmsf -s md_300K.tpr -f md_300K.xtc -ox average.pdb -o rmsf_all.xvg > average.out 2>&1 || tail -n 20 average.out
for ref in start last average; do
    case $ref in start) s=md_300K.tpr;; last) s=last.gro;; average) s=average.pdb;; esac
    printf "Backbone\nBackbone\n" | gmx rms -s $s -f md_300K.xtc -n md_300K.ndx -o rmsd_vs_$ref.xvg -tu ns > rms.out 2>&1 || tail -n 20 rms.out
done
ls rmsd_*.xvg

`-dump` writes the frame closest to the time given, so a very large time gives the last frame. `gmx rms` superimposes each frame on the reference using the first group, then computes the RMSD of the second.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for ref, label in {"start": "first frame", "last": "last frame", "average": "average structure"}.items():
    data = xvg(f"runs/rmsd_vs_{ref}.xvg")
    ax.plot(data[:, 0], data[:, 1], lw=0.8, label=label)
    print(f"{label:<18} mean {data[:, 1].mean():.3f} nm")
ax.set_xlabel("time (ns)")
ax.set_ylabel("backbone RMSD (nm)")
ax.legend(title="reference")
fig.tight_layout()
fig.savefig("results/rmsd_references.png", dpi=300)

<div class="alert alert-block alert-info">
Read the three curves. Why does each one start or end where it does? Which reference gives the lowest RMSD on average, and why? Open <code>runs/average.pdb</code> in VMD: is the average structure a structure the peptide could adopt? (Look at the side chains and the hydrogens.) What would you have concluded from the first curve alone?
</div>

## 2.1 The course's reference: the NMR structure

To compare runs and labs with each other, the course fixes **one** RMSD, used unchanged in LABs 5 and 6: the backbone (N, Cα, C) of residues 1–10, superimposed on and compared with **NMR model 1 of 2RVD** (`../common/structures/cln025_capped.pdb`). The atoms are matched by residue number and name, so the caps and the hydrogens do not matter. This is the function in `cln025.py`:

In [ ]:
print(inspect.getsource(cln025.atoms))
print(inspect.getsource(cln025.rmsd_nmr))

In [ ]:
import mdtraj as md

traj = md.load("runs/md_300K.xtc", top="runs/md_300K.pdb")
r = cln025.rmsd_nmr(traj)
print(f"{traj.n_frames} frames, RMSD from NMR model 1: mean {r.mean():.3f} nm, max {r.max():.3f} nm")

# 3. Flexibility and size

## 3.1 RMSF

The root-mean-square fluctuation of atom $i$ measures how much it moves around its average position over the trajectory ($T$ frames):

$$RMSF_i = \sqrt{\frac{1}{T}\sum_{t=1}^{T}\left|r_i(t) - \langle r_i\rangle\right|^2}$$

Computed per residue on the Cα atoms, it shows which parts of the peptide are rigid and which are floppy.

In [ ]:
%%bash
cd runs
echo CA | gmx rmsf -s md_300K.tpr -f md_300K.xtc -n md_300K.ndx -o rmsf.xvg -res > rmsf.out 2>&1 || tail -n 20 rmsf.out

In [ ]:
data = xvg("runs/rmsf.xvg")
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(data[:, 0], data[:, 1], "o-")
ax.set_xticks(range(1, 11), ["Y1", "Y2", "D3", "P4", "E5", "T6", "G7", "T8", "W9", "Y10"])
ax.set_ylabel("Cα RMSF (nm)")
fig.tight_layout()

<div class="alert alert-block alert-info">Which residues move most? Where would you expect a hairpin to fray first? Compare with the unfolding pathway described by Okumura (<i>Proteins</i> 2012, <a href="https://doi.org/10.1002/prot.24125">doi:10.1002/prot.24125</a>): the C-terminus, or both ends, open first.</div>

## 3.2 Radius of gyration

$$R_g = \sqrt{\frac{1}{N}\sum_{i=1}^{N}\left|r_i - r_{COM}\right|^2}$$

(mass-weighted in GROMACS) measures how compact the peptide is. Plot it in time and as a histogram:

In [ ]:
%%bash
cd runs
echo Protein | gmx gyrate -s md_300K.tpr -f md_300K.xtc -o rg.xvg > gyrate.out 2>&1 || tail -n 20 gyrate.out

In [ ]:
data = xvg("runs/rg.xvg")
fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4), gridspec_kw={"width_ratios": [2, 1]})
left.plot(data[:, 0] / 1000, data[:, 1], lw=0.7)
right.hist(data[:, 1], bins=40, density=True)
left.set_xlabel("time (ns)")
left.set_ylabel("$R_g$ (nm)")
right.set_xlabel("$R_g$ (nm)")
right.set_ylabel("probability density")
fig.tight_layout()
fig.savefig("results/radius_of_gyration.png", dpi=300)
print(f"R_g {data[:, 1].mean():.3f} ± {data[:, 1].std():.3f} nm")

# 4. Backbone angles: the Ramachandran plot

The conformation of each residue's backbone is set by two dihedral angles, φ and ψ. Their allowed combinations form the Ramachandran plot. `gmx rama` writes one line per residue per frame: φ, ψ and the residue name. The caps give Tyr1 and Tyr10 a full set of angles too.

In [ ]:
%%bash
cd runs
gmx rama -s md_300K.tpr -f md_300K.xtc -o rama.xvg > rama.out 2>&1 || tail -n 20 rama.out
grep -v "^[@#]" rama.xvg | head -n 10

The reference contours show where the backbones of 500 high-resolution protein structures fall (Lovell et al., *Proteins* 2003, [doi:10.1002/prot.10286](https://doi.org/10.1002/prot.10286)): the darker area holds 90% of them, the lighter 99%. The data ship with MDAnalysis.

In [ ]:
from MDAnalysis.analysis.data.filenames import Rama_ref


def rama_background(ax):
    X, Y = np.meshgrid(np.arange(-180, 180, 4), np.arange(-180, 180, 4))
    ax.contourf(X, Y, np.load(Rama_ref), levels=[1, 17, 15000], colors=["#A1D4FF", "#35A1FF"])
    ax.set_xlim(-180, 180)
    ax.set_ylim(-180, 180)
    ax.set_xlabel("φ (°)")
    ax.set_ylabel("ψ (°)")


phi, psi = np.loadtxt("runs/rama.xvg", comments=["#", "@"], usecols=(0, 1), unpack=True)
residues = np.loadtxt("runs/rama.xvg", comments=["#", "@"], usecols=2, dtype=str)
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
for ax, residue in zip(axes, ["ASP-3", "GLY-7", "TRP-9"]):
    rama_background(ax)
    keep = residues == residue
    ax.scatter(phi[keep], psi[keep], s=3, color="black")
    ax.set_title(residue)
fig.tight_layout()
fig.savefig("results/ramachandran.png", dpi=300)

**Gly7** is special. In the native hairpin it sits at positive φ (the left-handed, αL region), which glycine can reach because it has no side chain. A misfolded, out-of-register hairpin differs mainly in that Gly7 flips to negative φ (Kührová et al., *Biophys J* 2012, [doi:10.1016/j.bpj.2012.03.024](https://doi.org/10.1016/j.bpj.2012.03.024)). Keep an eye on Gly7 in sections 8 and 12.

<div class="alert alert-block alert-info">Which regions do Asp3 and Trp9 occupy, and which secondary structure do they correspond to? Where is Gly7, and does it stay there?</div>

# 5. Hydrogen bonds

`gmx hbond` finds hydrogen bonds from geometry: donor–acceptor distance up to 0.35 nm and hydrogen–donor–acceptor angle up to 30°. Count them inside the peptide, and between the peptide and water (for that, the full LAB 3 files with water are needed):

In [ ]:
%%bash
cd runs
gmx hbond -s md_300K.tpr -f md_300K.xtc -r protein -t protein -num hb_intra.xvg -tu ns > hbond.out 2>&1 || tail -n 20 hbond.out
src=../$LAB3/cln025_model1/04-md/md_300K
gmx hbond -s $src.tpr -f $src.xtc -r protein -t water -num hb_water.xvg -tu ns > hbond.out 2>&1 || tail -n 20 hbond.out

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, file, title in zip(axes, ["hb_intra", "hb_water"], ["inside the peptide", "peptide - water"]):
    data = xvg(f"runs/{file}.xvg")
    ax.plot(data[:, 0], data[:, 1], lw=0.7)
    ax.set_title(title)
    ax.set_xlabel("time (ns)")
    ax.set_ylabel("hydrogen bonds")
    print(f"{title:<19} {data[:, 1].mean():.1f} hydrogen bonds on average")
fig.tight_layout()
fig.savefig("results/hydrogen_bonds.png", dpi=300)

## 5.1 The three backbone hydrogen bonds of the hairpin

Three backbone hydrogen bonds hold the CLN025 hairpin together (you measured them in LAB 2): **Asp3 N–Thr8 O**, **Gly7 N–Asp3 O** and **Thr8 N–Asp3 O**. Follow each one through the run as the distance between the nitrogen and the oxygen, and count it as formed below 0.35 nm:

In [ ]:
%%bash
cd runs
gmx distance -s md_300K.tpr -f md_300K.xtc -oall hbonds3.xvg -tu ns > distance.out 2>&1 \
    -select 'resid 3 and name N plus resid 8 and name O' 'resid 7 and name N plus resid 3 and name O' \
            'resid 8 and name N plus resid 3 and name O' || tail -n 20 distance.out

In [ ]:
pairs = ["Asp3 N - Thr8 O", "Gly7 N - Asp3 O", "Thr8 N - Asp3 O"]
data = xvg("runs/hbonds3.xvg")
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8), sharey=True)
for i, (ax, pair) in enumerate(zip(axes, pairs)):
    ax.plot(data[:, 0], data[:, i + 1], lw=0.6)
    ax.axhline(0.35, color="black", ls="--", lw=0.8)
    ax.set_title(pair)
    ax.set_xlabel("time (ns)")
    print(f"{pair}: formed in {100 * (data[:, i + 1] < 0.35).mean():.0f}% of the frames")
axes[0].set_ylabel("N - O distance (nm)")
fig.tight_layout()

# 6. Secondary structure

<img src="imgs/SS.png" width="450">

*Figure: OpenStax, Biology 2e, CC BY 4.0.*

DSSP assigns a secondary structure to each residue from the pattern of backbone hydrogen bonds (Kabsch & Sander, *Biopolymers* 1983, [doi:10.1002/bip.360221211](https://doi.org/10.1002/bip.360221211)). `gmx dssp` writes one line per frame, one letter per residue:

| Letter | Structure | Letter | Structure |
|---|---|---|---|
| `E` | β-strand (in a ladder) | `H` | α-helix |
| `B` | isolated β-bridge | `G` | 3₁₀-helix |
| `T` | hydrogen-bonded turn | `I` | π-helix |
| `S` | bend | `P` | polyproline II |
| `~` | loop (none of these) | `=` | chain break |

The caps count as residues too, so each line has 12 letters, from ACE to NH2.

In [ ]:
%%bash
cd runs
gmx dssp -s md_300K.tpr -f md_300K.xtc -o dssp.dat -tu ns > dssp.out 2>&1 || tail -n 20 dssp.out
head -n 3 dssp.dat

In [ ]:
import matplotlib.colors as mcolors

codes = ["E", "B", "T", "S", "H", "G", "I", "P", "~", "="]
names = ["strand", "bridge", "turn", "bend", "α-helix", "3₁₀-helix", "π-helix", "PPII", "loop", "break"]
palette = ["#E15759", "#FF9DA7", "#4E79A7", "#76B7B2", "#F28E2B", "#EDC948", "#B07AA1", "#9C755F", "#EEEEEE", "#000000"]
cmap = mcolors.ListedColormap(palette)


def read_dssp(path):
    """DSSP letters of residues 1-10 (caps dropped), one row per frame."""
    return [line.strip()[1:-1] for line in open(path) if line.strip()]


frames = read_dssp("runs/dssp.dat")
grid = np.array([[codes.index(c) for c in f] for f in frames]).T
fig, ax = plt.subplots(figsize=(11, 3.5))
ax.imshow(grid, cmap=cmap, vmin=-0.5, vmax=len(codes) - 0.5, aspect="auto", interpolation="nearest")
ax.set_yticks(range(10), ["Y1", "Y2", "D3", "P4", "E5", "T6", "G7", "T8", "W9", "Y10"])
ax.set_xlabel("frame")
fig.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=c) for c in palette], labels=names, loc="center right")
fig.tight_layout(rect=(0, 0, 0.85, 1))
fig.savefig("results/secondary_structure.png", dpi=300)

strand = np.mean([f.count("E") + f.count("B") for f in frames])
print(f"residues in a strand or bridge per frame: {strand:.1f} of 10")

## 6.1 Two programs, one answer?

Older versions of GROMACS called an external DSSP program, and this course replaced it with MDTraj because the GROMACS route was unreliable. `gmx dssp`, rewritten in GROMACS 2023, has its own implementation. Check that the two agree on your trajectory. MDTraj has no polyproline II class, so `P` counts as loop for the comparison:

In [ ]:
mdtraj_dssp = md.compute_dssp(traj, simplified=False)[:, 1:-1]          # residues 1-10
gmx_dssp = np.array([list(f) for f in frames])
gmx_dssp[np.isin(gmx_dssp, ["~", "P"])] = " "     # MDTraj writes a loop as a space, and has no PPII class
n = min(len(gmx_dssp), len(mdtraj_dssp))
agree = (gmx_dssp[:n] == mdtraj_dssp[:n]).mean()
print(f"gmx dssp and MDTraj agree on {100 * agree:.1f}% of the residue assignments ({n} frames)")

# 7. Observables of the hairpin

Three numbers describe the state of CLN025 better than any general measure. Their definitions are fixed here and used unchanged in LABs 5 and 6.

1. **End-to-end distance**: between the Cα atoms of Tyr1 and Tyr10. About 0.5 nm in the folded hairpin; it is also the coordinate along which LAB 6 pulls.
2. **Aromatic core**: the distance between the centres of the Tyr2 and Trp9 rings, which stack against each other in the folded hairpin.
3. **Fraction of native contacts, Q**: the fraction of the contacts of the folded structure that are present, from 1 (all) to 0 (none).

In [ ]:
%%bash
cd runs
gmx distance -s md_300K.tpr -f md_300K.xtc -oall observables.xvg -tu ns > distance.out 2>&1 \
    -select 'resid 1 and name CA plus resid 10 and name CA' \
            'com of (resid 2 and name CG CD1 CD2 CE1 CE2 CZ) plus com of (resid 9 and name CD2 CE2 CE3 CZ2 CZ3 CH2)' \
    || tail -n 20 distance.out

Q follows Best, Hummer & Eaton (*PNAS* 2013, [doi:10.1073/pnas.1311599110](https://doi.org/10.1073/pnas.1311599110)): the native contacts are the pairs of heavy atoms, in residues more than three apart, that are closer than 0.45 nm in the folded reference structure (NMR model 1). Each contact counts smoothly, from 1 when it is at its native distance $r^0_{ij}$ to 0 when it is much longer:

$$Q(X) = \frac{1}{N}\sum_{(i,j)} \frac{1}{1 + \exp\left[\beta\left(r_{ij}(X) - \lambda\, r^0_{ij}\right)\right]}, \qquad \beta = 50\ \text{nm}^{-1},\ \lambda = 1.8$$

Only residues 1–10 are used, so the caps and the naming of the hydrogens do not matter. In `cln025.py`:

In [ ]:
print(inspect.getsource(cln025.native_contacts))
print(inspect.getsource(cln025.q_value))
print(f"{len(cln025.native_contacts())} native contacts")

In [ ]:
data = xvg("runs/observables.xvg")
q = cln025.q_value(traj)
fig, axes = plt.subplots(3, 1, figsize=(9, 8), sharex=True)
axes[0].plot(data[:, 0], data[:, 1], lw=0.6)
axes[1].plot(data[:, 0], data[:, 2], lw=0.6)
axes[2].plot(data[:, 0], q, lw=0.6)
axes[0].set_ylabel("end-to-end (nm)")
axes[1].set_ylabel("Tyr2 - Trp9 (nm)")
axes[2].set_ylabel("Q")
axes[2].set_xlabel("time (ns)")
fig.tight_layout()
fig.savefig("results/hairpin_observables.png", dpi=300)
print(f"end-to-end {data[:, 1].mean():.2f} nm, Tyr2-Trp9 {data[:, 2].mean():.2f} nm, Q {q.mean():.2f}")

<div class="alert alert-block alert-info">Compare the end-to-end distance with the 20 NMR models you measured in LAB 2. Do the three observables always agree on the state of the hairpin? If one of them moves while the others do not, what happened to the structure? Check the same moment in the three backbone hydrogen bonds of section 5.1, and in VMD.</div>

# 8. Many runs: the class dataset

One 10 ns run shows what one trajectory did. To say something about CLN025, you need many: the class dataset has one 300 K run for each NMR model of 2RVD. Download the course's copy in a terminal:

```bash
course-fetch --lab 04-Analysis --solutions
```

It lands in `solutions/class_amber/`, already prepared as in section 1.1 (`model01.xtc`, `model01.tpr`, `model01.pdb`, ...). To analyse your own class's runs instead, collect everybody's `cln025_model<N>` folders in `../03-ClassicalMD/class/` and run the next cell: it prepares them the same way, in `runs/class_amber/`. Then choose which set the rest of the lab reads: `solutions` for the course's, `runs` for your class's.

In [ ]:
%%bash
for src in ../03-ClassicalMD/class/cln025_model*/04-md/md_300K; do
    [ -f $src.xtc ] || continue
    model=$(echo $src | sed -E 's/.*cln025_model([0-9]+).*/\1/')
    name=runs/class_amber/model$(printf %02d $model)
    mkdir -p runs/class_amber
    printf "Protein\nProtein\n" | gmx trjconv -s $src.tpr -f $src.xtc -o $name.xtc -pbc mol -center > $name.out 2>&1
    printf "Protein\nProtein\n" | gmx trjconv -s $src.tpr -f $src.xtc -o $name.pdb -pbc mol -center -dump 0 >> $name.out 2>&1
    echo Protein | gmx convert-tpr -s $src.tpr -o $name.tpr >> $name.out 2>&1
    echo "prepared $name"
done

In [ ]:
%env CLASS=solutions

Load every run and compute the course's observables for each frame:

In [ ]:
import glob
import os

paths = sorted(glob.glob(f"{os.environ['CLASS']}/class_amber/model*.xtc"))
pooled = [cln025.observables(md.load(p, top=p.replace(".xtc", ".pdb"))) for p in paths]
models = [int(os.path.basename(p)[5:7]) for p in paths]
print(f"{len(pooled)} runs, {sum(len(obs['e2e']) for obs in pooled)} frames")

## 8.1 Is the hairpin folded?

A frame counts as **folded** when Q ≥ 0.7 and the backbone RMSD from the NMR structure is below 0.15 nm (`cln025.folded`). These thresholds are part of the course's fixed definitions, used again in LABs 5 and 6.

In [ ]:
fractions = np.array([cln025.folded(obs).mean() for obs in pooled])

fig, axes = plt.subplots(1, 4, figsize=(17, 4))
for ax, key, label, threshold in zip(axes, ["e2e", "q", "rmsd"], ["end-to-end (nm)", "Q", "RMSD from NMR (nm)"],
                                     [None, cln025.Q_FOLDED, cln025.RMSD_FOLDED]):
    ax.hist(np.concatenate([obs[key] for obs in pooled]), bins=60, density=True)
    if threshold is not None:
        ax.axvline(threshold, color="black", ls="--")
    ax.set_xlabel(label)
axes[0].set_ylabel("probability density")
axes[3].bar(models, fractions)
axes[3].set_xticks(models)
axes[3].tick_params(axis="x", labelsize=7)
axes[3].set_xlabel("NMR model the run started from")
axes[3].set_ylabel("folded fraction")
axes[3].set_ylim(0, 1.05)
fig.tight_layout()
fig.savefig("results/folded_class.png", dpi=300)

print(f"folded fraction {fractions.mean():.2f} ± {fractions.std(ddof=1) / np.sqrt(len(fractions)):.2f} "
      f"(mean ± standard error over {len(fractions)} runs; lowest run {fractions.min():.2f}, model {models[fractions.argmin()]})")

<div class="alert alert-block alert-info">
Is every run folded all the time? Which runs spend time outside the folded state, and what happened in them? Ten-nanosecond runs that all start folded can only show how stable the fold is over 10 ns, not the equilibrium population of the folded state: CLN025 folds and unfolds on a time scale of hundreds of nanoseconds or more. How long would the runs need to be to measure the population?
</div>

Keep in mind that the force field is a model, not the truth: in a comparison of four force fields on CLN025, AMBER ff14SB (a close relative of the course's amber99sb-ildn) stabilised the folded hairpin too much (Kamenik et al., *J Chem Phys* 2020, [doi:10.1063/5.0022135](https://doi.org/10.1063/5.0022135)).

# 9. Principal component analysis

The peptide's Cα atoms have 30 coordinates, but most of its motion happens along a few collective directions. **Principal component analysis** (PCA, or essential dynamics) finds them: `gmx covar` diagonalises the covariance matrix of the Cα positions, and its eigenvectors, sorted by eigenvalue, are the directions of largest motion. `gmx anaeig` then projects each frame on the first two.

Run the PCA on all the runs of the class dataset together, so that they share the same principal components:

In [ ]:
%%bash
cd runs
gmx trjcat -f ../$CLASS/class_amber/model*.xtc -o pca_all.xtc -cat > trjcat.out 2>&1 || tail -n 20 trjcat.out
printf "CA\nCA\n" | gmx covar -s md_300K.tpr -f pca_all.xtc -n md_300K.ndx -o eigenval.xvg -v eigenvec.trr > covar.out 2>&1 || tail -n 20 covar.out
printf "CA\nCA\n" | gmx anaeig -s md_300K.tpr -f pca_all.xtc -n md_300K.ndx -v eigenvec.trr \
                          -first 1 -last 2 -2d proj_pca_all.xvg > anaeig.out 2>&1 || tail -n 20 anaeig.out

`trjcat -cat` joins the runs one after the other, in the same order as the files were loaded in section 8, so each projected frame can be coloured by its Q:

In [ ]:
eigenval = xvg("runs/eigenval.xvg")
share = eigenval[:, 1] / eigenval[:, 1].sum()
print("variance captured by PC1, PC2, PC3:", ", ".join(f"{100 * s:.0f}%" for s in share[:3]))

proj = xvg("runs/proj_pca_all.xvg")
q_all = np.concatenate([obs["q"] for obs in pooled])
fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4.5))
left.bar(range(1, 11), 100 * share[:10])
left.set_xlabel("principal component")
left.set_ylabel("variance (%)")
points = right.scatter(proj[:, 0], proj[:, 1], s=2, c=q_all, cmap="viridis", vmin=0, vmax=1)
fig.colorbar(points, ax=right, label="Q")
right.set_xlabel("PC1 (nm)")
right.set_ylabel("PC2 (nm)")
fig.tight_layout()
fig.savefig("results/pca.png", dpi=300)

<div class="alert alert-block alert-info">How much of the motion do the first two components capture? Where are the frames with high Q, and where are the others? To see what PC1 means, <code>gmx anaeig -extr extreme.pdb -first 1 -last 1</code> writes the two extreme structures along it: open them in VMD.</div>

# 10. Free-energy surfaces

The probability $P(x, y)$ of finding the system at values $(x, y)$ of two observables defines a **free-energy surface**:

$$G(x, y) = -k_B T \ln P(x, y) + \text{constant}$$

Basins of low $G$ are the states the system visits often; high $G$ means rare. `gmx sham` builds it from a file with one column per observable. Take the end-to-end distance and Q, from all the runs of the class dataset:

In [ ]:
def write_columns(path, *columns):
    np.savetxt(path, np.column_stack(columns), fmt="%.4f")


e2e = np.concatenate([obs["e2e"] for obs in pooled])
q = np.concatenate([obs["q"] for obs in pooled])
rmsd = np.concatenate([obs["rmsd"] for obs in pooled])
write_columns("runs/q_e2e.xvg", q, e2e)

In [ ]:
%%bash
cd runs
gmx sham -f q_e2e.xvg -ls fes_q_e2e.xpm -notime -tsham 300 -ngrid 40 > sham.out 2>&1 || tail -n 20 sham.out

`gmx sham` writes the surface as an `.xpm` image, a text format in which every character stands for a colour and every colour for a value; `cln025.read_xpm` reads it back as numbers:

In [ ]:
x, y, G = cln025.read_xpm("runs/fes_q_e2e.xpm")
fig, ax = plt.subplots(figsize=(6, 4.5))
image = ax.pcolormesh(x, y, G, cmap="viridis")
fig.colorbar(image, label="G (kJ/mol)")
ax.set_xlabel("Q")
ax.set_ylabel("end-to-end distance (nm)")
ax.set_title("AMBER, 300 K, class dataset")
fig.tight_layout()
fig.savefig("results/fes_q_e2e.png", dpi=300)

The same surface in (RMSD, $R_g$), computed directly with NumPy to show that nothing more than a histogram is involved:

In [ ]:
kT = cln025.KT             # kJ/mol at 300 K
rg = np.concatenate([md.compute_rg(md.load(p, top=p.replace(".xtc", ".pdb"))) for p in paths])
H, xedges, yedges = np.histogram2d(rmsd, rg, bins=40)
with np.errstate(divide="ignore"):
    G2 = -kT * np.log(H.T / H.sum())
G2 -= np.nanmin(G2[np.isfinite(G2)])

fig, ax = plt.subplots(figsize=(6, 4.5))
image = ax.pcolormesh(xedges, yedges, np.ma.masked_invalid(G2), cmap="viridis", vmax=20)
fig.colorbar(image, label="G (kJ/mol)")
ax.set_xlabel("RMSD from NMR model 1 (nm)")
ax.set_ylabel("$R_g$ (nm)")
fig.tight_layout()
fig.savefig("results/fes_rmsd_rg.png", dpi=300)

<div class="alert alert-block alert-info">How many basins do you see? Empty (white) regions were never visited: their free energy is not infinite, just unknown. What limits how high a free energy you can measure from a histogram of <i>N</i> frames? (Hint: the rarest bin you can see holds one frame.)</div>

# 11. The equilibrium free-energy profile along the end-to-end distance

Reduce the surface to one coordinate, the end-to-end distance $d$, the coordinate along which LAB 6 pulls the peptide apart:

$$F(d) = -k_B T \ln P(d) + \text{constant}$$

Its minimum is set to zero. The error bars come from a **bootstrap over runs**: the profile is recomputed many times, each time from a random selection of the runs (with repetition), and the spread of the results is the uncertainty. Resampling whole runs, not frames, respects the fact that consecutive frames of one run are not independent.

In [ ]:
def profile(samples, edges):
    counts, _ = np.histogram(samples, bins=edges)
    with np.errstate(divide="ignore"):
        F = -kT * np.log(counts / counts.sum())
    return F - F[np.isfinite(F)].min(), counts


def pmf_bootstrap(runs_e2e, edges, n_boot=500, seed=1):
    rng = np.random.default_rng(seed)
    F, counts = profile(np.concatenate(runs_e2e), edges)
    boot = []
    for _ in range(n_boot):
        pick = rng.integers(0, len(runs_e2e), len(runs_e2e))
        Fb, _ = profile(np.concatenate([runs_e2e[i] for i in pick]), edges)
        boot.append(Fb)
    boot = np.array(boot)
    boot[~np.isfinite(boot)] = np.nan
    err = np.full(len(F), np.nan)
    seen = np.isfinite(boot).any(axis=0)          # bins reached by at least one bootstrap sample
    err[seen] = np.nanstd(boot[:, seen], axis=0)
    return F, err, counts


edges = np.arange(0.30, 2.51, 0.02)
centres = 0.5 * (edges[1:] + edges[:-1])
F, err, counts = pmf_bootstrap([obs["e2e"] for obs in pooled], edges)
ok = counts >= 5
np.savetxt("results/pmf_e2e_amber_300K.dat", np.column_stack([centres[ok], F[ok], err[ok], counts[ok]]),
           fmt="%.4f", header="d (nm)   F (kJ/mol)   bootstrap error (kJ/mol)   frames")
print(f"profile defined from {centres[ok].min():.2f} to {centres[ok].max():.2f} nm "
      f"(bins with at least 5 frames), highest point {F[ok].max():.1f} kJ/mol")

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.errorbar(centres[ok], F[ok], yerr=err[ok], capsize=2)
ax.set_xlabel("end-to-end distance d (nm)")
ax.set_ylabel("F(d) (kJ/mol)")
ax.set_title("amber99sb-ildn, 300 K, class dataset")
fig.tight_layout()
fig.savefig("results/pmf_e2e.png", dpi=300)

The profile is saved in `results/pmf_e2e_amber_300K.dat`: **LAB 6 overlays it on the free-energy profile from pulling**.

<div class="alert alert-block alert-info">
Over which range of distances is the profile defined, and why does it stop there? The unfolded hairpin should reach 2 nm and more: why does the profile not show it? What would you need to measure the free energy of the fully open hairpin from unbiased simulations? This limit is the reason for the pulling simulations of LAB 6.
</div>

# 12. Clustering the conformations

Clustering groups similar structures: `gmx cluster` with the GROMOS method (Daura et al., *Angew Chem Int Ed* 1999, [doi:10.1002/(SICI)1521-3773(19990115)38:1/2<236::AID-ANIE236>3.0.CO;2-M](https://doi.org/10.1002/(SICI)1521-3773(19990115)38:1/2%3C236::AID-ANIE236%3E3.0.CO;2-M)) takes the structure with the most neighbours within a cut-off RMSD, removes it and its neighbours as the first cluster, and repeats. Cluster the same frames as the PCA, every 50 ps, with a 0.1 nm backbone cut-off:

In [ ]:
%%bash
cd runs
printf "Backbone\nProtein\n" | gmx cluster -s md_300K.tpr -f pca_all.xtc -n md_300K.ndx -dt 50 \
    -method gromos -cutoff 0.1 -g cluster.log -cl clusters.pdb -sz cluster_size.xvg > cluster.out 2>&1 || tail -n 20 cluster.out
grep "Found" cluster.log

In [ ]:
sizes = xvg("runs/cluster_size.xvg")
centres_pdb = md.load("runs/clusters.pdb")
gly7 = [a.index for a in centres_pdb.topology.atoms
        if (a.residue.resSeq, a.name) in [(6, "C"), (7, "N"), (7, "CA"), (7, "C")]]
phi_gly7 = np.degrees(md.compute_dihedrals(centres_pdb, [gly7]))[:, 0]
q_centres = cln025.q_value(centres_pdb)
total = sizes[:, 1].sum()
print(f"{'cluster':>7} {'frames':>7} {'share':>6} {'Q':>5} {'Gly7 phi':>9}")
for i in range(min(8, len(sizes))):
    print(f"{int(sizes[i, 0]):7d} {int(sizes[i, 1]):7d} {sizes[i, 1] / total:6.2f} {q_centres[i]:5.2f} {phi_gly7[i]:9.0f}")

`clusters.pdb` holds the middle structure of each cluster: open it in VMD and look at the first few.

Maruyama & Mitsutake (*J Phys Chem B* 2018, [doi:10.1021/acs.jpcb.8b00288](https://doi.org/10.1021/acs.jpcb.8b00288)) classify chignolin's conformations into six states: two native, two misfolded, an intermediate and the unfolded state, and find the native and misfolded states almost equal in energy. The misfolded, out-of-register hairpin is recognised by Gly7, whose φ is negative instead of positive (Kührová et al. 2012, section 4).

<div class="alert alert-block alert-info">Which of your clusters are native? Is any misfolded (high Q is not enough: look at Gly7)? With 10 ns runs at 300 K that all start folded, which of the six states would you expect to miss, and why? LAB 5 heats the peptide to reach them.</div>

# 13. Your deliverable

1. **The RMSD with three references** of section 2, with a short answer to its question: what does an RMSD mean without its reference?
2. **The hairpin observables** of section 7 for your LAB 3 run, compared with the NMR models of LAB 2.
3. **The folded fraction** of the class dataset (section 8.1), with its error, and what it can and cannot tell you.
4. **A free-energy surface** (section 10) from the class dataset, with its basins labelled.
5. **The equilibrium profile** `results/pmf_e2e_amber_300K.dat` (section 11), with the range over which it is defined: you need it in LAB 6.
6. **A clustering report** (section 12): the largest clusters, their Q and Gly7 φ, and which ones are native, misfolded or unfolded. Finding a misfolded cluster as populated as the native one would be a real result.

As for the other simulation labs, present them in a few slides during a 15-minute group discussion.